In [16]:
import os
import json
import shutil
import random
from sklearn.model_selection import train_test_split

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
dataset_path = "/content/drive/MyDrive/tomato_project/archive/segmentation"

image_folder = os.path.join(dataset_path, "images")

annotation_file = os.path.join(dataset_path, "coco_annotations.json")

In [28]:
with open(annotation_file, "r") as f:
    coco_data = json.load(f)

In [22]:
import os

print(os.listdir("/content/drive/MyDrive/tomato_project"))

['archive']


In [24]:
print(os.listdir("/content/drive/MyDrive/tomato_project/archive/segmentation"))

['coco_annotations.json', 'labels', 'images']


In [32]:
import os

yolo_dataset_path = "/content/drive/MyDrive/tomato_project/yolo_dataset"

folders = [
    "images/train",
    "images/val",
    "images/test",
    "labels/train",
    "labels/val",
    "labels/test"
]

for folder in folders:
    os.makedirs(os.path.join(yolo_dataset_path, folder), exist_ok=True)

print("YOLO folder structure created successfully!")

YOLO folder structure created successfully!


In [33]:
images = coco_data["images"]

print("Total Images:", len(images))

Total Images: 177


In [34]:
train_images, temp_images = train_test_split(
    images,
    test_size=0.30,
    random_state=42,
    shuffle=True
)

print("Training Images:", len(train_images))
print("Temporary Images:", len(temp_images))

Training Images: 123
Temporary Images: 54


In [35]:
val_images, test_images = train_test_split(
    temp_images,
    test_size=1/3,
    random_state=42,
    shuffle=True
)

print("Training Images:", len(train_images))
print("Validation Images:", len(val_images))
print("Testing Images:", len(test_images))

Training Images: 123
Validation Images: 36
Testing Images: 18


In [36]:
image_dict = {}

for image in coco_data["images"]:
    image_dict[image["id"]] = image

In [37]:
def convert_to_yolo_segmentation(annotation, image_width, image_height):

    class_id = annotation["category_id"]

    segmentation = annotation["segmentation"][0]

    normalized_points = []

    for i in range(0, len(segmentation), 2):

        x = segmentation[i] / image_width
        y = segmentation[i + 1] / image_height

        normalized_points.append(f"{x:.6f}")
        normalized_points.append(f"{y:.6f}")

    yolo_line = str(class_id) + " " + " ".join(normalized_points)

    return yolo_line

In [38]:
def save_labels(image_list, split):

    for image in image_list:

        image_id = image["id"]

        image_name = image["file_name"]

        image_width = image["width"]

        image_height = image["height"]

        label_file = os.path.join(
            yolo_dataset_path,
            "labels",
            split,
            image_name.replace(".jpeg", ".txt")
        )

        with open(label_file, "w") as f:

            for annotation in coco_data["annotations"]:

                if annotation["image_id"] == image_id:

                    line = convert_to_yolo_segmentation(
                        annotation,
                        image_width,
                        image_height
                    )

                    f.write(line + "\n")

In [39]:
save_labels(train_images, "train")
save_labels(val_images, "val")
save_labels(test_images, "test")

print("YOLO labels created successfully!")

YOLO labels created successfully!


In [40]:
def copy_images(image_list, split):

    for image in image_list:

        src = os.path.join(
            image_folder,
            image["file_name"]
        )

        dst = os.path.join(
            yolo_dataset_path,
            "images",
            split,
            image["file_name"]
        )

        shutil.copy(src, dst)

In [41]:
copy_images(train_images, "train")
copy_images(val_images, "val")
copy_images(test_images, "test")

print("Images copied successfully!")

Images copied successfully!


In [42]:
yaml_text = f"""
path: {yolo_dataset_path}

train: images/train
val: images/val
test: images/test

names:
  0: unripe
  1: ripe
"""

with open(os.path.join(yolo_dataset_path, "data.yaml"), "w") as f:
    f.write(yaml_text)

print("data.yaml created successfully!")

data.yaml created successfully!


In [43]:
print("Train Images :", len(os.listdir(os.path.join(yolo_dataset_path, "images/train"))))
print("Validation Images :", len(os.listdir(os.path.join(yolo_dataset_path, "images/val"))))
print("Test Images :", len(os.listdir(os.path.join(yolo_dataset_path, "images/test"))))

print()

print("Train Labels :", len(os.listdir(os.path.join(yolo_dataset_path, "labels/train"))))
print("Validation Labels :", len(os.listdir(os.path.join(yolo_dataset_path, "labels/val"))))
print("Test Labels :", len(os.listdir(os.path.join(yolo_dataset_path, "labels/test"))))

Train Images : 123
Validation Images : 36
Test Images : 18

Train Labels : 123
Validation Labels : 36
Test Labels : 18


In [44]:
label_path = os.path.join(
    yolo_dataset_path,
    "labels",
    "train",
    os.listdir(os.path.join(yolo_dataset_path, "labels/train"))[0]
)

with open(label_path, "r") as f:
    print(f.read())

0 0.404132 0.134808 0.386422 0.125577 0.353311 0.121154 0.333804 0.125577 0.318018 0.135192 0.308393 0.137308 0.304928 0.144231 0.275796 0.172500 0.261679 0.195577 0.255005 0.213269 0.247305 0.245192 0.245637 0.283846 0.252695 0.336346 0.264887 0.369808 0.284394 0.400385 0.298255 0.413269 0.306597 0.417115 0.341889 0.382692 0.381032 0.361346 0.382957 0.324615 0.388090 0.289423 0.386679 0.277500 0.398742 0.248462 0.396176 0.223846 0.400411 0.216154 0.403619 0.224808 0.403106 0.231154 0.408111 0.234615 0.416324 0.233654 0.425565 0.220769 0.426976 0.199615 0.438013 0.184231 0.435190 0.173077 0.418121 0.148846
0 0.220739 0.420192 0.212012 0.418077 0.205210 0.430000 0.182367 0.453077 0.186473 0.458462 0.189040 0.470962 0.199435 0.484231 0.202387 0.496154 0.173896 0.469038 0.160164 0.465577 0.145662 0.467308 0.115118 0.488654 0.083932 0.498077 0.071997 0.535000 0.068147 0.563269 0.068403 0.588654 0.073280 0.622885 0.081622 0.652308 0.095098 0.684038 0.119867 0.723077 0.157983 0.756154 0.1864